# 💳 Credit Card Fraud Detection Using Machine Learning

## Project Overview

Credit card fraud is a major challenge for financial institutions because fraudulent transactions represent only a very small proportion of total transaction activity. This creates a difficult machine learning problem: a model can achieve extremely high accuracy by predicting that nearly every transaction is legitimate while still failing to identify fraudulent activity.

This project analyzes a large credit card transaction dataset and develops machine learning classification models to identify potentially fraudulent transactions.

The analysis will follow an end-to-end data science workflow, including data understanding, exploratory data analysis, data cleaning, feature engineering, preprocessing, model development, model evaluation, and interpretation.

---

## 🎯 Business Problem

Financial institutions process millions of credit card transactions, making manual identification of fraudulent activity impractical.

The primary business question for this project is:

> **Can transaction characteristics be used to identify potentially fraudulent credit card transactions while minimizing the number of fraudulent transactions that go undetected?**

Because fraudulent transactions are rare, special attention will be given to **class imbalance** and to evaluation metrics that measure the model's ability to detect fraud.

---

## 📊 Dataset

The dataset contains credit card transaction records with information including:

- Customer and card identifiers
- Transaction date and time
- Transaction amount
- Transaction method
- Merchant information
- Merchant location
- Merchant category code (MCC)
- Transaction errors
- Fraud status

The target variable for the machine learning problem is:

**`Is Fraud?`**

where each transaction is classified as either fraudulent or non-fraudulent.

---

## 🔍 Project Objectives

The objectives of this project are to:

1. Explore the structure and quality of the transaction data.
2. Investigate patterns associated with fraudulent transactions.
3. Clean and transform the data for machine learning.
4. Engineer meaningful features from transaction characteristics.
5. Address the severe imbalance between fraudulent and legitimate transactions.
6. Develop and compare multiple classification models.
7. Evaluate models using metrics appropriate for fraud detection.
8. Identify the model that provides the most useful balance between detecting fraud and limiting false alarms.
9. Translate model results into meaningful business insights and recommendations.

---

## 🧠 Modeling Considerations

Fraud detection is an **imbalanced classification problem**. Therefore, overall accuracy will not be used as the primary measure of model performance.

Model evaluation will emphasize:

- **Recall** — How much actual fraud did the model detect?
- **Precision** — Of the transactions flagged as fraud, how many were actually fraudulent?
- **F1-Score** — How well does the model balance precision and recall?
- **Confusion Matrix** — What types of classification errors is the model making?
- **Precision-Recall AUC** — How effectively does the model identify the rare positive class across classification thresholds?

Particular attention will be given to **false negatives**, since these represent fraudulent transactions that the model fails to detect.

---

## 🛠️ Project Workflow

**1. Business Understanding**  
Define the fraud detection problem and success criteria.

**2. Data Understanding**  
Examine dataset dimensions, variables, data types, missing values, and target distribution.

**3. Data Cleaning**  
Correct data types, investigate missing values, remove or address problematic observations, and prepare variables for analysis.

**4. Exploratory Data Analysis**  
Investigate transaction behavior and compare patterns between fraudulent and legitimate transactions.

**5. Feature Engineering**  
Create useful predictors from transaction amount, time, merchant information, location, transaction method, and other available variables.

**6. Preprocessing**  
Encode categorical variables, transform numerical variables where appropriate, and construct reproducible preprocessing pipelines.

**7. Model Development**  
Establish a baseline and train multiple classification algorithms.

**8. Imbalanced-Learning Strategy**  
Evaluate appropriate techniques for handling the highly imbalanced fraud target.

**9. Model Evaluation & Tuning**  
Compare models using recall, precision, F1-score, confusion matrices, and Precision-Recall performance.

**10. Model Interpretation**  
Investigate which transaction characteristics contribute most strongly to fraud predictions.

**11. Business Recommendations**  
Translate the analysis into actionable conclusions and discuss limitations and opportunities for future improvement.

## 📚 Import Libraries

The first step is to import the Python libraries needed for data manipulation, numerical analysis, and visualization.

- **OS** — Used to interact with the operating system and manage file paths.
- **Pandas** — Used for loading, cleaning, transforming, and analyzing tabular data.
- **NumPy** — Provides numerical computing tools and support for mathematical operations.
- **Matplotlib** — Used to create customizable data visualizations.
- **Seaborn** — Provides statistical visualization tools built on top of Matplotlib.

Additional libraries for preprocessing, machine learning, and model evaluation will be imported later as they are needed.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## 📥 Import the Dataset

The credit card transaction dataset is loaded into a Pandas DataFrame for analysis.

Each row represents an individual credit card transaction, while the columns contain information about the transaction, including the transaction date and time, amount, transaction method, merchant information, location, merchant category code, transaction errors, and fraud status.

The target variable for this project is **`Is Fraud?`**, which indicates whether a transaction was classified as fraudulent.

After importing the dataset, the first few rows are displayed to confirm that the data loaded successfully and to obtain an initial view of the dataset's structure.

In [ ]:
df = pd.read_csv(
    "credit_card_transactions-ibm_v2.csv"
)

df.head()

## 🔍 Checking for Missing Values

Before beginning data cleaning and exploratory analysis, the dataset is examined for missing values.

Missing data can affect statistical analysis, visualizations, feature engineering, and machine learning performance. However, a missing value does not always represent an error. In some variables, the absence of a value may carry meaningful information about the transaction.

The number of missing values in each column will first be identified. Columns containing substantial missing data will then be investigated individually before deciding whether values should be removed, replaced, or treated as a separate category.

In [ ]:
df.isnull().sum().sort_values(ascending=False)


### Missing Value Assessment

An initial assessment of missing values was conducted to evaluate the completeness of the transaction dataset.

Missing values were identified in three features: `Errors?`, `Zip`, and `Merchant State`. All remaining features contained no missing observations.

Before applying any preprocessing techniques, these missing values will be investigated to determine whether they represent unavailable data, meaningful transaction characteristics, or systematic patterns within the dataset. No features will be automatically dropped or imputed solely because missing values are present.

In [ ]:
missing_percentage = (df.isnull().sum() / len(df)) * 100

missing_percentage.sort_values(ascending=False)

In [ ]:
df["Errors?"].value_counts(dropna=False)

#### Errors Feature Investigation

The `Errors?` feature initially appeared to contain a very high percentage of missing values, with approximately 98.4% of transactions recorded as null. However, further investigation of the feature showed that the non-null observations represent specific transaction errors, including `Insufficient Balance`, `Bad PIN`, `Technical Glitch`, `Bad Card Number`, `Bad CVV`, and `Bad Expiration`.

Because specific error types are recorded when an error occurs, the null values likely represent transactions in which no error was recorded rather than traditional missing data.

Therefore, the null values in `Errors?` should not automatically be treated as a data-quality problem or used as a reason to remove the feature. During preprocessing, these null values can be explicitly represented as `No Error` so that the distinction between transactions with and without recorded errors is preserved.

### Investigating Missing Merchant Location Data

Missing values were identified in the `Merchant State` and `Zip` features. Rather than immediately imputing or removing these observations, the missingness will be investigated to determine whether it is associated with how the transaction was conducted.

Transactions with missing `Merchant State` values are filtered and compared with the `Use Chip` feature. This analysis will help determine whether missing merchant location information is associated with a particular transaction type, such as online transactions.

Understanding the reason for the missing values will help determine whether they represent a data-quality issue or meaningful information about the transaction.

In [ ]:
df[df["Merchant State"].isnull()]["Use Chip"].value_counts()

#### Merchant State Missingness Findings

The investigation shows that missing `Merchant State` values are overwhelmingly associated with online transactions. Of the transactions with missing merchant-state information, approximately 2.71 million were online transactions, compared with only about 7,600 chip transactions.

This suggests that the missing values are not primarily random data-quality issues. Instead, the absence of a merchant state appears to be associated with the transaction channel, particularly online purchases where a physical merchant location may not be applicable.

Therefore, the missing `Merchant State` values should not be automatically imputed with the most common state. Their relationship with online transactions should be considered during feature engineering and preprocessing.

#### Zip Code Missingness Investigation

The `Zip` feature contains approximately 11.8% missing values. Rather than immediately imputing or removing these observations, the missingness will be investigated to determine whether it is associated with the type of transaction.

Transactions with missing `Zip` values will be compared with the `Use Chip` feature to determine whether missing ZIP codes are primarily associated with online transactions. If a strong relationship exists, the missing values may represent meaningful transaction characteristics rather than random missing data.

The results of this investigation will be used to determine the appropriate preprocessing strategy for the `Zip` feature.

In [ ]:
df[df["Zip"].isnull()]["Use Chip"].value_counts()

In [ ]:
df[
    (df["Zip"].isnull()) &
    (df["Use Chip"] != "Online Transaction")
][["Merchant State", "Merchant City"]].head(10)

In [ ]:
df[
    (df["Zip"].isnull()) &
    (df["Use Chip"] != "Online Transaction")
]["Merchant State"].value_counts()

In [ ]:
df[
    (df["Zip"].isnull()) &
    (df["Use Chip"] != "Online Transaction")
]["Merchant State"]

#### Zip Code Missingness Findings

The `Zip` feature contains approximately 11.8% missing values. Investigation showed that most missing ZIP codes are associated with online transactions.

Further analysis of physical transactions with missing ZIP codes showed that these transactions were primarily associated with international merchant locations, including Mexico, Canada, Italy, the United Kingdom, and France. This indicates that missing ZIP codes are largely associated with transaction characteristics rather than random data-quality issues.

Therefore, missing `Zip` values should not be automatically imputed. Their relationship with online and international transactions should be preserved and considered during preprocessing and feature engineering.

### Target Variable Analysis

The target variable for this classification problem is `Is Fraud?`, which identifies whether each transaction was fraudulent or legitimate.

Before exploring relationships between fraud and the predictor variables, the distribution of the target variable will be examined. Understanding the class distribution is especially important in fraud detection because a large imbalance between legitimate and fraudulent transactions can affect model training and make certain evaluation metrics misleading.

The following analysis will examine:

- The number of legitimate and fraudulent transactions.
- The percentage of transactions belonging to each class.
- Whether class imbalance should influence the model evaluation strategy.

This assessment will help guide the selection of appropriate performance metrics during the modeling phase.

In [ ]:
df["Is Fraud?"].value_counts()

In [ ]:
df["Is Fraud?"].value_counts() / len(df) * 100

### Target Variable Distribution

The target variable, `Is Fraud?`, is highly imbalanced. Approximately 99.88% of transactions are legitimate, while only 0.12% are fraudulent.

This extreme class imbalance has important implications for model development and evaluation. A model that predicts every transaction as legitimate could achieve approximately 99.88% accuracy while failing to identify any fraudulent transactions.

Therefore, accuracy will not be used as the primary model-selection metric. Greater emphasis will be placed on metrics that evaluate performance on the minority fraud class, particularly recall and F2-score. Precision and the confusion matrix will also be monitored to evaluate the trade-off between detecting fraudulent transactions and incorrectly flagging legitimate transactions.

### Why Analyze Transaction Amount?

After examining the target variable and identifying the severe class imbalance between fraudulent and legitimate transactions, the next step was to investigate individual predictor variables that may contain useful information for identifying fraud.

`Amount` was selected as the first predictor because transaction size may differ between fraudulent and legitimate transactions. The initial hypothesis was that fraudulent transactions might involve smaller transaction amounts.

Rather than assuming this relationship existed, the `Amount` feature was analyzed to determine whether the data supported the hypothesis. This included examining its data type and distribution and comparing transaction amounts between the fraud and non-fraud classes.

The purpose of this analysis is to determine whether transaction amount shows a meaningful relationship with fraud and whether it may be useful as a predictor during model development.

### Preserving the Original Dataset

Before applying transformations to the transaction data, it is important to preserve the original dataset in its raw form. This provides a reference to the data as it was initially loaded and prevents exploratory preprocessing steps from permanently altering the original values.

A separate working version of the dataset will be used for data cleaning, exploratory analysis, feature engineering, and modeling. For example, the `Amount` feature is currently stored as an object because the transaction values contain a dollar sign. The working dataset can be updated by removing the dollar sign and converting `Amount` to a numeric data type while leaving the original data unchanged.

Because this dataset contains approximately 24.4 million transactions, maintaining multiple complete copies in memory may require substantial RAM. Therefore, copies of the full dataset should be created only when necessary, and memory usage will be considered throughout the analysis.

In [ ]:
df_original = df.copy()

df_clean = df.copy()


In [ ]:
df_clean["Amount"] = (
    df_clean["Amount"]
    .str.replace("$", "", regex=False)
    .astype(float)
)

In [ ]:
df_clean["Amount"].head(10)


In [ ]:
df_clean["Amount"].dtype

In [ ]:
df_clean["Amount"].describe()

In [ ]:
df_clean.groupby("Is Fraud?")["Amount"].agg(["mean", "median"])

In [ ]:
plt.figure(figsize=(8, 6))

sns.boxplot(
    data=df_clean,
    x="Is Fraud?",
    y="Amount"
)

plt.title("Transaction Amount by Fraud Status")
plt.xlabel("Fraud Status")
plt.ylabel("Transaction Amount ($)")
plt.show()

### Investigating Negative Transaction Amounts

Before applying a logarithmic transformation to the `Amount` feature, the presence of negative transaction values must be investigated.

A log transformation such as `log1p(x)` calculates the logarithm of `1 + x` for each transaction amount. This transformation can help reduce the effect of extreme positive values and make a positively skewed distribution easier to analyze. However, `log1p(x)` cannot be directly applied to values less than -1 because the logarithm would be undefined in the real-number system.

The initial box plot revealed that the dataset contains negative transaction amounts. These values may represent refunds, returns, reversals, credits, or another type of transaction rather than data errors. Therefore, they should not be automatically removed or transformed without further investigation.

The next step is to examine the frequency and distribution of negative transaction amounts before deciding how they should be handled during preprocessing and visualization.

In [ ]:
df_clean[df_clean["Amount"] < 0]["Amount"].describe()

In [ ]:
(df_clean["Amount"] < 0).sum()

In [ ]:
df_clean[df_clean["Amount"] < 0][["Amount", "Is Fraud?"]]

In [ ]:
df_clean[(df_clean["Amount"] < 0)][["Is Fraud?"]].value_counts(normalize=True)

#### Negative Transaction Amount Findings

The dataset contains approximately 1.24 million transactions with negative `Amount` values. Among these transactions, approximately 0.09% were fraudulent, compared with an overall fraud rate of approximately 0.12% across the full dataset.

This indicates that negative transaction amounts have a slightly lower fraud rate than the dataset overall and do not appear to be a strong fraud indicator on their own. However, negative amounts may still contain useful predictive information when considered alongside other transaction characteristics.

Because negative transactions represent a substantial portion of the dataset and include both legitimate and fraudulent observations, they should not be removed simply to accommodate a logarithmic transformation. An alternative approach for handling the skewness of `Amount` should therefore be considered while preserving these observations.

In [ ]:
df_clean["Amount"].quantile(0.99)

In [ ]:
sns.boxplot(data=df_clean, x="Is Fraud?", y="Amount")

plt.ylim(-317, 317)

plt.show()

### Transaction Amount Findings

The analysis of `Amount` revealed meaningful differences between fraudulent and legitimate transactions. Fraudulent transactions had a median amount of approximately **$71.02**, compared with **$30.11** for legitimate transactions. Fraudulent transactions also displayed a larger interquartile range, indicating greater variability within the middle 50% of transaction amounts.

The distribution of transaction amounts was strongly right-skewed and contained numerous extreme values. The 99th percentile of `Amount` was approximately **$316.99**. To improve the readability of the box plot, the displayed y-axis range was restricted while preserving all observations in the underlying dataset.

The box plot showed that fraudulent transactions generally have higher transaction amounts and greater variability than legitimate transactions. However, substantial overlap exists between the two groups. Therefore, transaction amount alone would not be sufficient to distinguish fraudulent from legitimate transactions reliably.

Overall, `Amount` appears to contain useful information for fraud detection and should be retained as a potential predictor. Its predictive value should ultimately be evaluated in combination with other transaction characteristics during model development.

### Transaction Method Analysis

After examining transaction amount, the next step is to investigate whether the method used to complete a transaction is associated with fraud.

The `Use Chip` feature identifies how each transaction was conducted, including chip, swipe, and online transactions. Different transaction methods may have different levels of exposure to fraudulent activity, making this feature potentially useful for fraud detection.

The analysis will first examine the distribution of transactions across the different transaction methods. The fraud rate within each transaction method will then be compared to determine whether certain methods are more strongly associated with fraudulent transactions.

This analysis will help determine whether `Use Chip` may provide useful predictive information for the fraud classification model.

In [ ]:
df_clean["Use Chip"].value_counts()

In [ ]:
df_clean.groupby("Use Chip")["Is Fraud?"].value_counts(normalize=True)


### Transaction Method Findings

The `Use Chip` feature was analyzed to determine whether the method used to complete a transaction is associated with fraudulent activity.

Swipe transactions were the most common transaction type in the dataset, followed by chip transactions, while online transactions were the least common. However, the distribution of fraudulent transactions showed a different pattern. Online transactions accounted for **18,349 fraudulent transactions**, compared with **6,572 swipe transactions** and **4,836 chip transactions**.

Because the number of transactions differs substantially across transaction methods, fraud rates were compared rather than relying only on raw fraud counts. Online transactions had a fraud rate of approximately **0.676%**, compared with approximately **0.077% for chip transactions** and **0.043% for swipe transactions**.

The online transaction fraud rate is also substantially higher than the overall dataset fraud rate of approximately **0.12%**, representing roughly **5.6 times the overall rate**.

These results indicate a strong association between transaction method and fraudulent activity in this dataset. In particular, online transactions appear substantially more likely to be associated with fraud than chip or swipe transactions. This does not imply that conducting a transaction online causes fraud, but it suggests that `Use Chip` may provide valuable predictive information and should be retained for further analysis and modeling.

### Transaction Error Analysis

The `Errors?` feature records whether a transaction was associated with a specific transaction error, such as insufficient balance, bad PIN, technical glitch, bad card number, or other recorded errors.

Earlier analysis showed that approximately 98.4% of the values in `Errors?` are null. However, because the non-null observations represent specific error types, the null values likely indicate transactions where no error was recorded rather than traditional missing data.

The next step is to investigate whether the presence of a recorded transaction error is associated with fraudulent activity. The analysis will first determine how many transactions contain a recorded error and then compare fraud patterns between transactions with and without recorded errors.

This analysis will help determine whether the presence of a transaction error, and potentially specific error types, may provide useful predictive information for fraud detection.

In [ ]:
len(df_clean[df_clean["Errors?"].notnull()])

In [ ]:
df_clean[df_clean["Errors?"].notnull()]["Is Fraud?"].value_counts()

In [ ]:
df_clean[df_clean["Errors?"].notnull()]["Is Fraud?"].value_counts(normalize=True)

In [ ]:
df_clean[df_clean["Errors?"].notnull()] .groupby("Errors?")["Is Fraud?"].value_counts(normalize=True)

In [ ]:
df_clean[df_clean["Errors?"].notnull()]["Errors?"].value_counts()


### Transaction Error Findings

The `Errors?` feature was investigated to determine whether recorded transaction errors are associated with fraudulent activity.

Approximately 388,000 transactions contained a recorded error. Transactions with recorded errors had a fraud rate of approximately **0.33%**, compared with an overall fraud rate of approximately **0.12%**. This suggests that the presence of a transaction error may contain useful information for identifying fraudulent transactions.

Further analysis showed that fraud rates varied substantially across specific error types. In particular, transactions associated with `Bad CVV` had a fraud rate of approximately **2.61%**, which is substantially higher than the overall fraud rate. Because the `Bad CVV` category contains approximately **10,740 transactions**, this pattern is supported by a considerably larger sample than some of the less common error combinations.

Some rare combinations displayed even higher fraud rates. For example, `Bad Expiration, Technical Glitch` had a fraud rate of approximately **6.25%**, but this category contained only **32 transactions**. Therefore, the high percentage should be interpreted cautiously because a small number of observations can produce unstable fraud-rate estimates.

Overall, the analysis suggests that `Errors?`, particularly certain error types such as `Bad CVV`, may contain useful predictive information. However, a feature should not be included in a predictive model solely because it is strongly associated with the target variable.

Before using `Errors?` during modeling, the timing of this information must be considered. If transaction-error information is available at the time the fraud prediction is made, it may be a valid predictor. If the error information is generated only after the prediction or transaction outcome, using it could introduce data leakage. Therefore, `Errors?` will be retained as a candidate feature while its availability at prediction time is evaluated.

In [ ]:
df_clean.info()


### Transaction Time Analysis

Transaction time may contain useful information for identifying patterns associated with fraudulent activity. Fraudulent transactions may occur more frequently during certain hours of the day, making time a potentially useful predictive feature.

The `Time` variable is currently stored as an object in `HH:MM` 24-hour format. Analyzing every individual minute would create too many categories and make patterns difficult to interpret. Therefore, the hour of the transaction will be extracted into a separate `Hour` feature while preserving the original `Time` variable.

The analysis will investigate:

- How transactions are distributed across the 24 hours of the day.
- Whether fraud rates vary by hour.
- Whether particular hours show unusually high fraud activity.
- Whether observed differences are supported by sufficient transaction volume rather than small sample sizes.

This analysis will help determine whether transaction hour may provide useful information for fraud detection.

In [ ]:
df_clean["Time"].head(5)


In [ ]:
df_clean["Hour"]=df_clean["Time"].str[0:2].astype(int)

In [ ]:
df_clean[["Time", "Hour"]].head(5)

In [ ]:
df_clean.shape


In [ ]:
df_clean.groupby("Hour")["Is Fraud?"].value_counts(normalize=True)

In [ ]:
df_clean.groupby("Hour")["Is Fraud?"].value_counts()

In [ ]:
df_clean["Hour"].value_counts()

In [ ]:
fraud_by_hour = df_clean.groupby("Hour")["Is Fraud?"].value_counts(normalize=True)

In [ ]:
fraud_by_hour.xs("Yes", level="Is Fraud?").sort_values()

In [ ]:
len(df_clean[df_clean["Hour"] == 22])

### Transaction Hour Findings

Transaction hour appears to be associated with differences in fraud activity across the day.

The highest observed hourly fraud rate occurred around **10 AM (Hour 10)**, where approximately **0.1872%** of transactions were fraudulent. This is higher than the overall dataset fraud rate of approximately **0.122%**. Approximately **1.66 million transactions** occurred during Hour 10, indicating that the elevated fraud rate is supported by a substantial number of observations.

In contrast, **10 PM (Hour 22)** had the lowest observed fraud rate at approximately **0.0116%**. Approximately **759,960 transactions** occurred during this hour, providing a substantial sample size for the observed rate. The surrounding late-evening hours (Hours 20–23) also displayed fraud rates below the overall dataset fraud rate, suggesting a broader time-of-day pattern rather than an isolated result.

These findings demonstrate an association between transaction hour and fraud occurrence but do not establish that the time of day causes fraudulent activity.

Based on these results, `Hour` will be retained as a candidate feature for predictive modeling because fraud rates vary meaningfully across different hours of the day and the observed patterns are supported by substantial transaction volumes.

Because hour-of-day is cyclical, however, treating `Hour` as an ordinary numerical variable may not fully represent its structure. For example, Hour 23 and Hour 0 are only one hour apart on a clock despite having numerical values that differ by 23. Therefore, cyclical encoding of the `Hour` feature will be considered during feature engineering.

In [ ]:
df_clean["Hour_sin"] = np.sin(2 * np.pi * df_clean["Hour"] / 24)

In [ ]:
df_clean["Hour_cos"] = np.cos(2 * np.pi * df_clean["Hour"] / 24)

In [ ]:
df_clean[["Hour","Hour_sin","Hour_cos"]].head(10)

In [ ]:
df_clean[df_clean["Hour"].isin([0, 23])][["Hour", "Hour_sin", "Hour_cos"]]

In [ ]:
df_clean[df_clean["Hour"].isin([0, 23])][["Hour", "Hour_sin", "Hour_cos"]].drop_duplicates()

### Cyclical Encoding of Transaction Hour

Although the `Hour` feature represents the time of day, treating hour as an ordinary numerical variable does not accurately represent the relationship between different times.

For example, Hours 23 and 0 represent 11 PM and midnight. Numerically, their difference is:

\[
|23 - 0| = 23
\]

However, these times are actually only one hour apart. This occurs because time follows a **cyclical structure**, where the end of one 24-hour period connects directly to the beginning of the next.

To preserve this circular relationship, the `Hour` feature was transformed using sine and cosine:

\[
Hour_{sin} = \sin\left(\frac{2\pi \times Hour}{24}\right)
\]

\[
Hour_{cos} = \cos\left(\frac{2\pi \times Hour}{24}\right)
\]

Since a complete circle contains \(2\pi\) radians and the day contains 24 hours, each hour represents:

\[
\frac{2\pi}{24} = \frac{\pi}{12}
\]

radians, or 15 degrees, around the unit circle.

The sine and cosine transformations represent each hour using two coordinates on the unit circle. Both coordinates are necessary because a single sine or cosine value may correspond to more than one position on the circle.

The transformation was verified by comparing Hours 23 and 0:

| Hour | Hour_sin | Hour_cos |
|---:|---:|---:|
| 23 | -0.258819 | 0.965926 |
| 0 | 0.000000 | 1.000000 |

Although the original numerical values 23 and 0 appear far apart, their transformed coordinates are close together because Hour 23 corresponds to 345 degrees and Hour 0 corresponds to 0/360 degrees. These positions are only 15 degrees apart on the unit circle.

This representation therefore preserves the natural cyclical structure of transaction time and provides a more meaningful representation of hour-of-day for models that are sensitive to numerical distance.

The original `Hour` feature will also be retained during development so that the usefulness of the raw and cyclical representations can be evaluated during model validation rather than assuming that cyclical encoding will improve every model.

### Merchant Category Code (MCC) Analysis

Merchant Category Code (`MCC`) identifies the type of business or merchant category associated with a transaction. Because different types of merchants may experience different levels of fraud risk, `MCC` may provide useful information for identifying fraudulent transactions.

Before comparing fraud rates across merchant categories, the structure of the `MCC` feature will first be examined. In particular, the analysis will determine:

- How many unique MCC categories are represented in the dataset.
- How transactions are distributed across the different MCC categories.
- Whether certain MCC categories have substantially higher fraud rates than others.
- Whether high observed fraud rates are supported by sufficient transaction volume rather than small sample sizes.

The number of unique MCC values is also important for feature engineering. If `MCC` contains many categories, treating it as an ordinary numerical variable may incorrectly imply that the numerical distance between MCC codes has meaningful quantitative information. The number and frequency of categories will therefore be considered when determining how `MCC` should eventually be represented for predictive modeling.

As with previous analyses, both the **fraud rate and the number of transactions within each category** will be considered before concluding.

In [ ]:
df_clean["MCC"].nunique()

In [ ]:
df_clean["MCC"].value_counts()


In [ ]:
df_clean.groupby("MCC")["Is Fraud?"].value_counts(normalize=True)

In [ ]:
df_clean.groupby("MCC")["Is Fraud?"].value_counts(normalize=True).xs("Yes", level="Is Fraud?")

In [ ]:
df_clean["MCC"].value_counts()

In [ ]:
len(df_clean[df_clean["MCC"] == 3006])

In [ ]:
mcc_summary = df_clean.groupby("MCC").agg(
    total_transactions=("MCC", "size")
)

In [ ]:
mcc_summary 

In [ ]:
fraud_counts = (
    df_clean[df_clean["Is Fraud?"] == "Yes"]
    .groupby("MCC")
    .size()
    .rename("fraud_transactions")
)

In [ ]:
fraud_counts

In [ ]:
mcc_summary = mcc_summary.join(fraud_counts, how="left")

In [ ]:
mcc_summary

In [ ]:
df.info()


In [ ]:
df.isna().sum()

In [ ]:
df["Errors?"].value_counts(dropna=False)


In [ ]:
df[df["Is Fraud?"] == "Yes"]["Errors?"].notnull().sum()

In [ ]:
df[df["Is Fraud?"] == "Yes"]["Errors?"].notnull().sum() / df["Errors?"].notnull().sum() * 100